# 간호사 행동/도메인 특화 VLM – LoRA Fine-tuning 과제

## 과제 목표
NurViD 간호 행위 영상에서 추출한 이미지를 활용하여:
1. **Baseline**: SmolVLM zero-shot caption 생성 및 평가 (BLEU / METEOR / CIDEr-D)
2. **Fine-tuning**: LoRA로 간호 특화 VLM 학습
3. **Before / After 비교**: 의료 object·action 인식 개선 분석

## 진행 흐름
```
NurViD 영상 다운로드
  → 이미지 프레임 추출
  → GT Caption 생성 (annotation 기반)
  → Baseline zero-shot 평가
  → LoRA Fine-tuning
  → Fine-tuned 평가
  → Before/After 비교 분석
```

---
## 0. 환경 설정 및 라이브러리 설치

In [ ]:
# Colab 환경 설정
# Colab에서 Drive를 mount하고 프로젝트 경로를 import path에 추가합니다.
# Drive I/O가 느릴 때는 아래 setup_colab_workdir 주석을 해제해 /content 복사본 기준으로 실행할 수 있습니다.
import importlib.util
import sys

DRIVE_PROJECT = '/content/drive/MyDrive/VLM'  # 본인 Drive 폴더명에 맞게 수정

if importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive

    drive.mount('/content/drive')
    if DRIVE_PROJECT not in sys.path:
        sys.path.insert(0, DRIVE_PROJECT)

# Drive I/O가 느릴 때만 사용한다.
# gd_mount: Colab Drive 프로젝트를 로컬 작업 경로로 복사하고 import path를 맞춥니다.
# from utils.gd_mount import setup_colab_workdir
# setup_colab_workdir(drive_project=DRIVE_PROJECT, mount_drive=False)


Mounted at /content/drive


In [ ]:
# 필요 라이브러리 설치
!pip install -q \
    transformers==4.57.6 \
    datasets==4.8.4 \
    evaluate==0.4.6 \
    nltk==3.9.4 \
    accelerate==1.13.0 \
    peft==0.18.1 \
    Pillow==11.3.0 \
    tqdm==4.67.3 \
    yt-dlp \
    pycocoevalcap \
    matplotlib \
    seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.0 MB/s eta 0:00:00


In [ ]:
# GPU 확인
import subprocess
import torch

try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi를 찾을 수 없습니다. GPU 런타임인지 확인하세요.')

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 device: {DEVICE}')

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB
사용 device: cuda


In [ ]:
# 공통 import
import os
import json
import random
import subprocess
from pathlib import Path
from dataclasses import dataclass
from typing import Optional
from collections import Counter
from functools import partial
from itertools import cycle

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)

print('Import 완료')

Import 완료


In [ ]:
# 재현성을 위한 시드 고정
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Seed {SEED}로 고정')

Seed 42로 고정


---
## 1. NurViD 데이터셋 준비

### 1-1. 작업 디렉터리 설정

In [ ]:
import os, sys
from pathlib import Path

# ── 기존 프로젝트 (utils 등 코드 참조용, 읽기 전용) ──────────────────────────
VLM_DIR = Path('/content/drive/MyDrive/VLM')
if str(VLM_DIR) not in sys.path:
    sys.path.insert(0, str(VLM_DIR))

# ── 내 작업 공간 (데이터/결과 저장용) ────────────────────────────────────────
NURSE_DIR = Path('/content/drive/MyDrive/Nurse')
DATA_DIR  = NURSE_DIR / 'data'
VIDEO_DIR = DATA_DIR  / 'videos'
FRAME_DIR = DATA_DIR  / 'frames'
MODEL_DIR = NURSE_DIR / 'model'

# annotation은 Nurse 폴더 바로 아래
ANNO_PATH = NURSE_DIR / 'NurViD_annotations.json'

# ── 디렉터리 생성 ─────────────────────────────────────────────────────────────
for d in [VIDEO_DIR, FRAME_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── 확인 ─────────────────────────────────────────────────────────────────────
print(f'[VLM 프로젝트]  {VLM_DIR}  {"✅" if VLM_DIR.exists() else "❌ 없음"}')
print(f'[Nurse 작업공간]{NURSE_DIR} {"✅" if NURSE_DIR.exists() else "❌ 없음"}')
print(f'[Annotation]    {ANNO_PATH} {"✅" if ANNO_PATH.exists() else "❌ 없음"}')
print()
print('[생성된 디렉터리]')
for d in [VIDEO_DIR, FRAME_DIR, MODEL_DIR]:
    print(f'  {"✅" if d.exists() else "❌"} {d}')

[VLM 프로젝트]  /content/drive/MyDrive/VLM  ✅
[Nurse 작업공간]/content/drive/MyDrive/Nurse ✅
[Annotation]    /content/drive/MyDrive/Nurse/NurViD_annotations.json ✅

[생성된 디렉터리]
  ✅ /content/drive/MyDrive/Nurse/data/videos
  ✅ /content/drive/MyDrive/Nurse/data/frames
  ✅ /content/drive/MyDrive/Nurse/model


### 1-2. NurViD Annotation 로드

GitHub에서 `NurViD_annotations.json`을 받거나, 이미 있으면 `DATA_DIR`에 두세요.

In [ ]:
# annotation 로드 (Nurse 폴더에 이미 업로드되어 있음)
with open(ANNO_PATH, 'r', encoding='utf-8') as f:
    annotations = json.load(f)

print(f'✅ annotation 로드 완료: {ANNO_PATH}')
print(f'총 영상 수: {len(annotations)}개')

sample_key = list(annotations.keys())[0]
print(f'\n샘플 미리보기 (key: {sample_key}):')
print(json.dumps(annotations[sample_key], indent=2, ensure_ascii=False)[:400])

✅ annotation 로드 완료: /content/drive/MyDrive/Nurse/NurViD_annotations.json
총 영상 수: 1538개

샘플 미리보기 (key: --Ly-qjodoI):
{
  "operationID": 3,
  "annotations": [
    {
      "actionID": 29,
      "segment": [
        10.78,
        18.51
      ]
    },
    {
      "actionID": 4,
      "segment": [
        187.61,
        200.15
      ]
    },
    {
      "actionID": 7,
      "segment": [
        200.51,
        253.9
      ]
    },
    {
      "actionID": 1,
      "segment": [
        316.71,
        366.75
      ]



In [ ]:
# Procedure / Action 매핑 (Procedure&Action_ID.xlsx 0-indexed + 1 = JSON ID)
PROCEDURE_MAP = {
    5: 'Closed Intravenous Infusion',
    6: 'Subcutaneous Injection',
    8: 'Blood Glucose Monitoring',
}

ACTION_MAP = {
    1:  'Disinfect skin',
    2:  'Handwashing',
    3:  'Position the patient',
    4:  'Check',
    5:  'Inject medication',
    6:  'Place an underpad',
    7:  'Venipuncture',
    8:  'Cleanse skin',
    9:  'Rinse with running water',
    10: 'Document',
    12: 'Perform surgical hand scrub',
    14: 'Prepare medication solution',
    15: 'Release trapped air',
    16: 'Select a vein',
    18: 'Remove needle',
    19: 'Perform subcutaneous puncture',
    20: 'Measure blood glucose level',
    22: 'Connect infusion device',
    25: 'Prepare glucometer',
    31: 'Establish a sterile zone',
    37: 'Put on isolation gown',
    42: 'Aspirate medication',
    43: 'Assist patient taking medicine',
    63: 'Cleanse lips',
    75: 'Defibrillate',
    94: 'Adjust drip rate',
    102:'Connect suction catheter',
}

TARGET_PROCEDURES = set(PROCEDURE_MAP.keys())

print('선정 Procedure:')
for pid, name in PROCEDURE_MAP.items():
    print(f'  operationID {pid}: {name}')

선정 Procedure:
  operationID 5: Closed Intravenous Infusion
  operationID 6: Subcutaneous Injection
  operationID 8: Blood Glucose Monitoring


### 1-3. GT Caption 생성

Caption 규칙: **role + action + medical object + hospital context**

예: `"A nurse preparing a syringe in a hospital ward"`

In [ ]:
def build_gt_caption(procedure_name: str, action_name: str) -> str:
    action_to_phrase = {
        # 기존
        'Disinfect skin':                'disinfecting the patient\'s skin',
        'Handwashing':                   'washing hands before the procedure',
        'Position the patient':          'positioning the patient on the bed',
        'Check':                         'checking the medication and patient information',
        'Inject medication':             'injecting medication into the patient',
        'Place an underpad':             'placing an underpad beneath the patient',
        'Venipuncture':                  'performing venipuncture on the patient\'s arm',
        'Cleanse skin':                  'cleansing the skin at the procedure site',
        'Rinse with running water':      'rinsing the area with running water',
        'Document':                      'documenting the procedure on patient records',
        'Perform surgical hand scrub':   'performing a surgical hand scrub',
        'Prepare medication solution':   'preparing the medication solution',
        'Release trapped air':           'releasing trapped air from the IV line',
        'Select a vein':                 'selecting a suitable vein for injection',
        'Remove needle':                 'safely removing the needle after the procedure',
        'Perform subcutaneous puncture': 'performing a subcutaneous puncture',
        'Measure blood glucose level':   'measuring the patient\'s blood glucose level',
        'Connect infusion device':       'connecting the infusion device to the IV line',
        'Prepare glucometer':            'preparing the glucometer for blood glucose testing',
        'Establish a sterile zone':      'establishing a sterile zone for the procedure',
        'Aspirate medication':           'aspirating medication into the syringe',
        'Adjust drip rate':              'adjusting the IV drip rate',
        'Put on isolation gown':         'putting on an isolation gown',
        # 새로 추가
        # action_to_phrase에 추가
        'Assist patient taking medicine': 'assisting the patient in taking medicine',
        'Secure ostomy bag':             'securing the ostomy bag in place',
        'Select a vein':                 'selecting a suitable vein for injection',
        'Connect infusion device':       'connecting the infusion device to the IV line',
        'Prepare glucometer':            'preparing the glucometer for blood glucose testing',
        'Adjust drip rate':              'adjusting the IV drip rate',
        'Select needle puncture site':   'selecting the needle puncture site',
    }

    action_phrase = action_to_phrase.get(
        action_name,
        f'performing {action_name.lower()}'
    )
    return (
        f"A nurse {action_phrase} "
        f"as part of a {procedure_name.lower()} procedure "
        f"in a hospital ward."
    )


# 테스트
print(build_gt_caption('Closed Intravenous Infusion', 'Venipuncture'))
print(build_gt_caption('Blood Glucose Monitoring', 'Handwashing'))
print(build_gt_caption('Subcutaneous Injection', 'Aspirate medication'))

A nurse performing venipuncture on the patient's arm as part of a closed intravenous infusion procedure in a hospital ward.
A nurse washing hands before the procedure as part of a blood glucose monitoring procedure in a hospital ward.
A nurse aspirating medication into the syringe as part of a subcutaneous injection procedure in a hospital ward.


### 1-4. 영상 다운로드 및 프레임 추출

> **주의:** 전체 NurViD는 100GB+입니다. `MAX_VIDEOS`를 환경에 맞게 조절하세요.

In [ ]:
# ─── 영상 다운로드 함수 (url 필드 사용) ─────────────────────────────────────

def download_video(url: str, video_id: str, output_dir: Path) -> Optional[Path]:
    """
    yt-dlp로 YouTube 영상 다운로드.
    video_id = YouTube ID (예: '25DJ1kvaN60') → 파일명으로 사용
    url = 'https://www.youtube.com/watch?v=25DJ1kvaN60'
    """
    out_path = output_dir / f'{video_id}.mp4'
    if out_path.exists():
        return out_path  # 이미 존재하면 스킵

    cmd = [
        'yt-dlp',
        '-f', 'bestvideo[ext=mp4][height<=480]+bestaudio[ext=m4a]/best[ext=mp4][height<=480]/best',
        '--merge-output-format', 'mp4',
        '-o', str(output_dir / f'{video_id}.%(ext)s'),
        '--quiet', '--no-warnings',
        url,
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
        if result.returncode == 0 and out_path.exists():
            return out_path
    except Exception:
        pass
    return None


def extract_frames_from_segment(
    video_path: Path,
    frame_dir: Path,
    video_id: str,
    ann_idx: int,
    segment: list,          # [start_sec, end_sec] ← 실제 JSON 형식
    fps: float = 0.5,
    max_frames: int = 5,
) -> list:
    """
    ffmpeg으로 segment = [start, end] 구간에서 프레임 추출.
    저장 경로: frame_dir / video_id / ann{idx:02d} / frame_XXXX.jpg
    """
    start, end = segment[0], segment[1]
    duration   = max(end - start, 1.0)

    seg_dir = frame_dir / video_id / f'ann{ann_idx:02d}'
    seg_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        'ffmpeg', '-y',
        '-ss', str(start),
        '-t',  str(duration),
        '-i',  str(video_path),
        '-vf', f'fps={fps}',
        '-frames:v', str(max_frames),
        '-q:v', '2',
        '-loglevel', 'error',
        str(seg_dir / 'frame_%04d.jpg'),
    ]
    try:
        subprocess.run(cmd, capture_output=True, timeout=60)
    except Exception:
        pass

    return sorted(seg_dir.glob('frame_*.jpg'))


print('다운로드 / 프레임 추출 함수 정의 완료')

다운로드 / 프레임 추출 함수 정의 완료


In [ ]:
# ─── 영상 필터링 (실제 JSON 구조 기반) ──────────────────────────────────────
#
# 구조 재확인:
#   data[video_id] = {
#     'operationID': 6,
#     'annotations': [{'actionID': 2, 'segment': [19.75, 28.67]}, ...],
#     'url': 'https://www.youtube.com/watch?v=...',
#     ...
#   }
#
# → 영상 1개에 여러 (actionID, segment) 쌍이 있으므로
#   각 annotation 단위로 프레임 추출 + caption 생성

with open(ANNO_PATH, 'r', encoding='utf-8') as f:
    annotations = json.load(f)

# Target Procedure 필터링
filtered = {
    vid_id: item
    for vid_id, item in annotations.items()
    if item['operationID'] in TARGET_PROCEDURES
}

print(f'전체 영상:              {len(annotations)}개')
print(f'선정 Procedure 해당:    {len(filtered)}개')
print()

from collections import Counter
proc_dist = Counter(
    PROCEDURE_MAP[item['operationID']]
    for item in filtered.values()
)
for name, cnt in proc_dist.most_common():
    print(f'  {name}: {cnt}개')

전체 영상:              1538개
선정 Procedure 해당:    177개

  Closed Intravenous Infusion: 60개
  Subcutaneous Injection: 60개
  Blood Glucose Monitoring: 57개


In [ ]:
MAX_VIDEOS         = 177   # 전체 다 사용
FPS_EXTRACT        = 0.5
MAX_FRAMES_PER_ANN = 5

dataset_items = []
failed_videos = []

# Procedure별 균등 샘플링
proc_buckets = {}
for vid_id, item in filtered.items():
    proc_buckets.setdefault(item['operationID'], []).append((vid_id, item))

selected = []
per_proc = max(1, MAX_VIDEOS // len(proc_buckets))
for op_id, vid_list in proc_buckets.items():
    selected.extend(vid_list[:per_proc])
selected = selected[:MAX_VIDEOS]
random.shuffle(selected)

print(f'다운로드 대상: {len(selected)}개 영상')
print(f'설정: FPS={FPS_EXTRACT}, annotation당 최대 {MAX_FRAMES_PER_ANN}프레임\n')

for vid_id, item in tqdm(selected, desc='영상 처리'):
    url          = item['url']
    op_id        = item['operationID']
    proc_name    = PROCEDURE_MAP[op_id]
    video_annots = item['annotations']   # [{actionID, segment}, ...]

    # 영상 다운로드
    video_path = download_video(url, vid_id, VIDEO_DIR)
    if video_path is None:
        failed_videos.append(vid_id)
        continue

    # annotation 단위로 프레임 추출 (화이트리스트 없이 전체 사용)
    for ann_idx, ann in enumerate(video_annots):
        action_id   = ann['actionID']
        segment     = ann['segment']          # [start, end]
        action_name = ACTION_MAP.get(action_id, f'action_{action_id}')
        gt_caption  = build_gt_caption(proc_name, action_name)

        frames = extract_frames_from_segment(
            video_path, FRAME_DIR, vid_id, ann_idx, segment,
            fps=FPS_EXTRACT, max_frames=MAX_FRAMES_PER_ANN,
        )

        for fp in frames:
            dataset_items.append({
                'image_path': str(fp),
                'caption':    gt_caption,
                'procedure':  proc_name,
                'action':     action_name,
                'video_id':   vid_id,
                'action_id':  action_id,
                'segment':    segment,
            })

print(f'\n구축된 데이터셋: {len(dataset_items)}개 이미지-caption 쌍')
print(f'다운로드 실패:   {len(failed_videos)}개')
if failed_videos:
    print(f'실패 목록: {failed_videos}')

In [ ]:
# dataset.json 다시 저장
with open(NURSE_DIR / 'data' / 'dataset.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_items, f, ensure_ascii=False, indent=2)
print(f'데이터셋 저장 완료: {len(dataset_items)}개')

# Train / Val / Test Split 재설정 (8:1:1)
random.shuffle(dataset_items)
n       = len(dataset_items)
n_train = int(n * 0.8)
n_val   = int(n * 0.1)

train_items = dataset_items[:n_train]
val_items   = dataset_items[n_train:n_train + n_val]
test_items  = dataset_items[n_train + n_val:]

print(f'Train: {len(train_items)}, Val: {len(val_items)}, Test: {len(test_items)}')
print('\nCaption 예시:')
for item in train_items[:3]:
    print(f'  [{item["procedure"]}]\n  → {item["caption"]}')

NameError: name 'dataset_items' is not defined

---
## 2. Metric 계산 유틸리티

In [ ]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import numpy as np # Add import for numpy


def compute_bleu(predictions: list, references: list) -> dict:
    """BLEU-1 ~ BLEU-4 계산."""
    smoothie = SmoothingFunction().method1
    refs_tok = [[ref.lower().split()] for ref in references]
    hyps_tok = [pred.lower().split() for pred in predictions]
    scores = {}
    for n in range(1, 5):
        weights = tuple([1.0/n]*n + [0.0]*(4-n))
        scores[f'BLEU-{n}'] = corpus_bleu(refs_tok, hyps_tok,
                                           weights=weights,
                                           smoothing_function=smoothie)
    return scores


def compute_meteor(predictions: list, references: list) -> float:
    """평균 METEOR 점수 계산."""
    return float(np.mean([
        meteor_score([ref.lower().split()], pred.lower().split())
        for pred, ref in zip(predictions, references)
    ]))


def compute_cider(predictions: list, references: list) -> float:
    """CIDEr-D 점수 계산 (pycocoevalcap 사용)."""
    try:
        from pycocoevalcap.cider.cider import Cider
        score, _ = Cider().compute_score(
            {i: [ref]  for i, ref  in enumerate(references)},
            {i: [pred] for i, pred in enumerate(predictions)}
        )
        return float(score)
    except ImportError:
        print('pycocoevalcap 없음 → CIDEr-D = 0.0')
        return 0.0


def compute_all_metrics(predictions: list, references: list) -> dict:
    """BLEU / METEOR / CIDEr-D 통합 계산."""
    return {
        **compute_bleu(predictions, references),
        'METEOR':  compute_meteor(predictions, references),
        'CIDEr-D': compute_cider(predictions, references)
    }


print('Metric 함수 정의 완료')

Metric 함수 정의 완료


---
## 3. SmolVLM 로드 및 Caption 생성 함수

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = 'HuggingFaceTB/SmolVLM-256M-Instruct'

# 간호 특화 prompt: role + action + medical object + hospital context 유도
NURSING_PROMPT = (
    'Describe this medical image in one sentence. '
    'Include the nurse role, the medical action being performed, '
    'and any medical equipment visible.'
)


def load_smolvlm(model_id: str = MODEL_ID, device=DEVICE):
    """SmolVLM 모델과 processor를 로드합니다."""
    print(f'모델 로드 중: {model_id}')
    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16 if device.type == 'cuda' else torch.float32,
    ).to(device)
    model.eval()
    if hasattr(model, 'generation_config') and processor.tokenizer.eos_token_id:
        model.generation_config.eos_token_id = processor.tokenizer.eos_token_id
        model.generation_config.pad_token_id = processor.tokenizer.eos_token_id
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'파라미터: {total:,} total / {trainable:,} trainable')
    return model, processor


def generate_caption(model, processor, image_path: str,
                     prompt: str = NURSING_PROMPT,
                     max_new_tokens: int = 64, device=DEVICE) -> str:
    """
    단일 이미지에 대한 caption을 생성합니다.
    SmolVLM의 chat template 형식을 사용합니다.
    """
    image = Image.open(image_path).convert('RGB')
    messages = [{
        'role': 'user',
        'content': [{'type': 'image'}, {'type': 'text', 'text': prompt}],
    }]
    text   = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors='pt').to(device)

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    generated = out_ids[:, inputs['input_ids'].shape[1]:]
    return processor.decode(generated[0], skip_special_tokens=True).strip()


def generate_captions_batch(model, processor, items: list,
                             prompt: str = NURSING_PROMPT,
                             max_new_tokens: int = 64, device=DEVICE) -> list:
    """items 리스트에 대해 caption을 일괄 생성합니다."""
    preds = []
    for item in tqdm(items, desc='Caption 생성'):
        try:
            preds.append(generate_caption(model, processor, item['image_path'],
                                          prompt=prompt, max_new_tokens=max_new_tokens,
                                          device=device))
        except Exception:
            preds.append('')
    return preds

print('Caption 생성 함수 정의 완료')

Caption 생성 함수 정의 완료


---
## 4. Baseline: SmolVLM Zero-shot 평가

In [ ]:
# SmolVLM base 모델 로드
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
base_model, processor = load_smolvlm(MODEL_ID, DEVICE)

모델 로드 중: HuggingFaceTB/SmolVLM-256M-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

파라미터: 256,484,928 total / 256,484,928 trainable


In [ ]:
# 프레임 이미지 추출 확인
from pathlib import Path

frame_files = list(FRAME_DIR.rglob('*.jpg'))
print(f'추출된 프레임 수: {len(frame_files)}개')

# dataset_items의 image_path가 실제로 존재하는지 확인
missing = [i for i in dataset_items if not Path(i['image_path']).exists()]
print(f'dataset_items 중 파일 없는 것: {len(missing)}개')

# 샘플 확인
print('\n프레임 샘플 경로:')
for f in frame_files[:3]:
    print(f'  {f}')

KeyboardInterrupt: 

In [ ]:
# Zero-shot Caption 생성 (test split)
print(f'Zero-shot 평가 대상: {len(test_items)}개 이미지')
zs_predictions = generate_captions_batch(base_model, processor, test_items,
                                          prompt=NURSING_PROMPT, max_new_tokens=64, device=DEVICE)
zs_references  = [item['caption'] for item in test_items]

print('\n=== Zero-shot Caption 예시 ===')
for i in range(min(5, len(test_items))):
    print(f'\n[이미지 {i+1}] {test_items[i]["procedure"]} / {test_items[i]["action"]}')
    print(f'  GT:        {zs_references[i]}')
    print(f'  Zero-shot: {zs_predictions[i]}')

In [ ]:
# Zero-shot Metric 계산
print('Zero-shot Metric 계산 중...')
zs_metrics = compute_all_metrics(zs_predictions, zs_references)

print('\n=== Zero-shot 평가 결과 ===')
for metric, score in zs_metrics.items():
    print(f'  {metric:10s}: {score:.4f}')

In [ ]:
# ─── 의료 Object/Action 실패 분석 ────────────────────────────────────────
#
# 분석 항목:
#   1. 의료 object 인식 실패 (syringe, IV, glucometer 등)
#   2. 의료 행동 collapse (injecting → "holding something")
#   3. hallucination 발생 여부

MEDICAL_OBJECTS = ['syringe','needle','iv','drip','glucometer','stethoscope',
                   'glove','bandage','tubing','catheter','strip','monitor']
MEDICAL_ACTIONS = ['inject','infus','monitor','nurse','nursing','medical',
                   'venipuncture','glucose','medication','hospital','patient']

def analyze_failure(predictions: list, references: list) -> dict:
    obj_fail = action_fail = halluc = 0
    collapse_words = ['holding','standing','sitting','looking','person','man','woman']
    for pred, ref in zip(predictions, references):
        pl, rl = pred.lower(), ref.lower()
        if any(o in rl for o in MEDICAL_OBJECTS) and not any(o in pl for o in MEDICAL_OBJECTS):
            obj_fail += 1
        if any(c in pl for c in collapse_words) and not any(a in pl for a in MEDICAL_ACTIONS):
            action_fail += 1
        if any(o in pl and o not in rl for o in MEDICAL_OBJECTS):
            halluc += 1
    n = max(len(predictions), 1)
    return {
        'total': n,
        'medical_object_fail': obj_fail,    'medical_object_fail_pct':    round(obj_fail/n*100,1),
        'medical_action_collapse': action_fail,'medical_action_collapse_pct':round(action_fail/n*100,1),
        'hallucination': halluc,            'hallucination_pct':           round(halluc/n*100,1),
    }

zs_failure = analyze_failure(zs_predictions, zs_references)
print('=== Zero-shot 실패 분석 ===')
print(f'  총 샘플:                  {zs_failure["total"]}개')
print(f'  의료 Object 인식 실패:    {zs_failure["medical_object_fail"]}개 ({zs_failure["medical_object_fail_pct"]}%)')
print(f'  의료 행동 Collapse:       {zs_failure["medical_action_collapse"]}개 ({zs_failure["medical_action_collapse_pct"]}%)')
print(f'  Hallucination 발생:       {zs_failure["hallucination"]}개 ({zs_failure["hallucination_pct"]}%)')

In [ ]:
# Zero-shot 결과 저장 + 메모리 정리
zs_result = {
    'metrics':     zs_metrics,
    'failure':     zs_failure,
    'predictions': [{'image': item['image_path'], 'gt': ref, 'pred': pred,
                     'procedure': item['procedure'], 'action': item['action']}
                    for item, ref, pred in zip(test_items, zs_references, zs_predictions)]
}
with open(MODEL_DIR / 'zero_shot_results.json', 'w', encoding='utf-8') as f:
    json.dump(zs_result, f, ensure_ascii=False, indent=2)
print('Zero-shot 결과 저장 완료')

del base_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('GPU 메모리 정리 완료')

In [ ]:
with open(MODEL_DIR / 'zero_shot_results.json', 'r') as f:
    zs_result = json.load(f)
zs_metrics = zs_result['metrics']
print(zs_metrics)

{'BLEU-1': 0.17379592030577717, 'BLEU-2': 0.05488505217502689, 'BLEU-3': 0.017435619956127076, 'BLEU-4': 0.0037694415042091632, 'METEOR': 0.10175980256837941, 'CIDEr-D': 0.013219105915010444}


In [ ]:
import os
size = os.path.getsize(MODEL_DIR / 'zero_shot_results.json')
print(f'파일 크기 확인: {size:,} bytes')

파일 크기 확인: 68,797 bytes


더 보기
오후 3:54
Zero-shot 결과가 잘 나왔어요! 과제 분석 관점에서 보면:

예상대로 SmolVLM이 간호 도메인에 취약한 게 확인됐어요:

GT:        A nurse cleansing the skin at the procedure site...
Zero-shot: Person is getting a procedure with a blue band around their wrist.
→ 의료 행동 완전 collapse, "nurse" 인식 실패

GT:        A nurse washing hands before the procedure...  
Zero-shot: A person is using an item to make a medical appointment.
→ "medical appointment" hallucination 발생

GT:        A nurse measuring the patient's blood glucose level...
Zero-shot: A person is getting a medical procedure on their stomach.
→ 혈당측정을 "stomach procedure"로 오인
수치 정리:

의료 Object 인식 실패: 29.1%
의료 행동 Collapse: 22.4%
Hallucination: 21.8%
BLEU-4: 0.0035 (거의 0에 가까움)

---
## 5. LoRA Fine-tuning

### 5-1. Dataset 클래스 및 Collate 함수

In [ ]:
from torch.utils.data import Dataset, DataLoader

from PIL import Image

class NursingCaptionDataset(Dataset):
    def __init__(self, items, processor, prompt=NURSING_PROMPT):
        self.items, self.processor, self.prompt = items, processor, prompt

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item  = self.items[idx]
        # 이미지 리사이즈로 토큰 수 줄이기
        image = Image.open(item['image_path']).convert('RGB')
        image = image.resize((224, 224), Image.LANCZOS)  # 강제 224x224

        messages = [
            {'role': 'user',
             'content': [{'type': 'image'}, {'type': 'text', 'text': self.prompt}]},
            {'role': 'assistant',
             'content': [{'type': 'text', 'text': item['caption']}]},
        ]
        text = self.processor.apply_chat_template(messages, add_generation_prompt=False)
        return {'image': image, 'text': text, 'caption': item['caption']}
def nursing_collate_fn(batch, processor, device):
    images   = [b['image']   for b in batch]
    texts    = [b['text']    for b in batch]
    captions = [b['caption'] for b in batch]

    inputs = processor(
        text=texts,
        images=images,
        return_tensors='pt',
        padding=True,
        truncation=False,
        image_seq_len=64,   # 이미지 토큰 수 직접 제한
    ).to(device)

    labels = inputs['input_ids'].clone()
    for i, caption in enumerate(captions):
        cap_ids = processor.tokenizer(caption, add_special_tokens=False)['input_ids']
        prompt_len = labels.shape[1] - len(cap_ids)
        if prompt_len > 0:
            labels[i, :prompt_len] = -100

    labels[inputs['attention_mask'] == 0] = -100
    inputs['labels'] = labels
    return inputs
print('Dataset / Collate 정의 완료')

Dataset / Collate 정의 완료


### 5-2. LoRA 설정 및 모델 준비

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType


import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

@dataclass
class LoRAConfig:
    model_id: str = MODEL_ID
    rank: int        = 16
    alpha: int       = 32
    dropout: float   = 0.05
    target_modules: tuple = (
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    )
    num_epochs: int  = 3
    batch_size: int  = 1      # 4 → 1
    grad_accum: int  = 16     # 4 → 16 (실효 batch 유지)
    lr: float        = 2e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    max_new_tokens: int = 64
    output_dir: Path    = MODEL_DIR

lora_cfg = LoRAConfig()
print(f'rank={lora_cfg.rank}, alpha={lora_cfg.alpha}, lr={lora_cfg.lr}')
print(f'target_modules={lora_cfg.target_modules}')

rank=16, alpha=32, lr=0.0002
target_modules=('q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj')


In [ ]:
import gc
import torch

# 가능한 모든 모델 변수 삭제
for var in ['ft_model', 'base_model', 'ft_model_eval', 'ft_processor', 'processor', 'eval_processor']:
    if var in dir():
        exec(f'del {var}')

# 전체 객체 순회하며 모델 찾아서 삭제
import transformers
import peft
for name, obj in list(globals().items()):
    if isinstance(obj, (transformers.PreTrainedModel, peft.PeftModel)):
        print(f'삭제: {name}')
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f'사용 중 VRAM: {torch.cuda.memory_allocated(0)/1e9:.1f} GB')
print(f'여유 VRAM: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1e9:.1f} GB')

사용 중 VRAM: 1.2 GB
여유 VRAM: 14.4 GB


In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

def load_smolvlm(model_id=MODEL_ID, device=DEVICE):
    print(f'모델 로드 중: {model_id}')
    processor = AutoProcessor.from_pretrained(model_id)

    # 이미지 패치 수 제한 (13 → 1)
    if hasattr(processor, 'image_processor'):
        processor.image_processor.max_image_tiles = 1
        processor.image_processor.size = {"longest_edge": 512}

    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16 if device.type == 'cuda' else torch.float32,
    ).to(device)
    model.eval()
    if hasattr(model, 'generation_config') and processor.tokenizer.eos_token_id:
        model.generation_config.eos_token_id = processor.tokenizer.eos_token_id
        model.generation_config.pad_token_id = processor.tokenizer.eos_token_id
    return model, processor

# 기존 모델 정리 후 재로드
del ft_model, ft_processor
gc.collect()
torch.cuda.empty_cache()

ft_model, ft_processor = load_smolvlm(lora_cfg.model_id, DEVICE)

# 패치 수 확인
image = Image.open(train_items[0]['image_path']).convert('RGB')
messages = [
    {'role': 'user',
     'content': [{'type': 'image'}, {'type': 'text', 'text': NURSING_PROMPT}]},
]
text   = ft_processor.apply_chat_template(messages, add_generation_prompt=False)
inputs = ft_processor(text=[text], images=[image], return_tensors='pt')
print(f'pixel_values 크기: {inputs["pixel_values"].shape}')
print(f'input_ids 길이: {inputs["input_ids"].shape[1]}')

모델 로드 중: HuggingFaceTB/SmolVLM-256M-Instruct
pixel_values 크기: torch.Size([1, 1, 3, 512, 512])
input_ids 길이: 97


In [ ]:
# SmolVLM 재로드 + LoRA 적용
ft_model, ft_processor = load_smolvlm(lora_cfg.model_id, DEVICE)

# PEFT LoraConfig 정의
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,        # 언어 모델 causal LM task
    r=lora_cfg.rank,                      # low-rank 행렬 차원
    lora_alpha=lora_cfg.alpha,            # scaling factor
    lora_dropout=lora_cfg.dropout,        # dropout
    target_modules=list(lora_cfg.target_modules),
    bias='none',                          # bias는 학습하지 않음
    inference_mode=False,
)

# base model에 LoRA adapter 삽입 → 원래 가중치 freeze, A/B 행렬만 학습
ft_model = get_peft_model(ft_model, peft_config)
ft_model.print_trainable_parameters()

모델 로드 중: HuggingFaceTB/SmolVLM-256M-Instruct
trainable params: 5,769,216 || all params: 262,254,144 || trainable%: 2.1999


In [ ]:
# 패치 수 확인
image = Image.open(train_items[0]['image_path']).convert('RGB')
messages = [
    {'role': 'user',
     'content': [{'type': 'image'}, {'type': 'text', 'text': NURSING_PROMPT}]},
]
text   = ft_processor.apply_chat_template(messages, add_generation_prompt=False)
inputs = ft_processor(text=[text], images=[image], return_tensors='pt')
print(f'pixel_values 크기: {inputs["pixel_values"].shape}')
print(f'input_ids 길이: {inputs["input_ids"].shape[1]}')

pixel_values 크기: torch.Size([1, 1, 3, 512, 512])
input_ids 길이: 97


In [ ]:
with open(MODEL_DIR / 'zero_shot_results.json', 'r') as f:
    zs_result = json.load(f)
zs_metrics      = zs_result['metrics']
zs_failure      = zs_result['failure']
zs_predictions  = [p['pred'] for p in zs_result['predictions']]
zs_references   = [p['gt']   for p in zs_result['predictions']]
test_items      = zs_result['predictions']
print('Zero-shot 결과 로드 완료')
print(f'BLEU-4: {zs_metrics["BLEU-4"]:.4f}')

Zero-shot 결과 로드 완료
BLEU-4: 0.0038


### 5-3. 학습 루프

In [ ]:
import json
import random
from pathlib import Path

NURSE_DIR = Path('/content/drive/MyDrive/Nurse')
DATA_DIR  = NURSE_DIR / 'data'
FRAME_DIR = DATA_DIR  / 'frames'
MODEL_DIR = NURSE_DIR / 'model'
ANNO_PATH = NURSE_DIR / 'NurViD_annotations.json'

with open(ANNO_PATH, 'r', encoding='utf-8') as f:
    annotations = json.load(f)

PROCEDURE_MAP = {
    5: 'Closed Intravenous Infusion',
    6: 'Subcutaneous Injection',
    8: 'Blood Glucose Monitoring',
}

ACTION_MAP = {
    1:'Disinfect skin', 2:'Handwashing', 3:'Position the patient',
    4:'Check', 5:'Inject medication', 6:'Place an underpad',
    7:'Venipuncture', 8:'Cleanse skin', 9:'Rinse with running water',
    10:'Document', 11:'Secure ostomy bag', 12:'Perform surgical hand scrub',
    14:'Prepare medication solution', 15:'Release trapped air',
    16:'Select a vein', 17:'Select a vein', 18:'Remove needle',
    19:'Perform subcutaneous puncture', 20:'Measure blood glucose level',
    22:'Connect infusion device', 23:'Connect infusion device',
    25:'Prepare glucometer', 26:'Prepare glucometer',
    31:'Establish a sterile zone', 37:'Put on isolation gown',
    42:'Aspirate medication', 43:'Assist patient taking medicine',
    63:'Cleanse lips', 91:'Wear gloves', 94:'Adjust drip rate',
    95:'Adjust drip rate', 117:'Select needle puncture site',
}

def build_gt_caption(procedure_name, action_name):
    action_to_phrase = {
        'Disinfect skin':                'disinfecting the patient\'s skin',
        'Handwashing':                   'washing hands before the procedure',
        'Position the patient':          'positioning the patient on the bed',
        'Check':                         'checking the medication and patient information',
        'Inject medication':             'injecting medication into the patient',
        'Place an underpad':             'placing an underpad beneath the patient',
        'Venipuncture':                  'performing venipuncture on the patient\'s arm',
        'Cleanse skin':                  'cleansing the skin at the procedure site',
        'Rinse with running water':      'rinsing the area with running water',
        'Document':                      'documenting the procedure on patient records',
        'Secure ostomy bag':             'securing the ostomy bag in place',
        'Perform surgical hand scrub':   'performing a surgical hand scrub',
        'Prepare medication solution':   'preparing the medication solution',
        'Release trapped air':           'releasing trapped air from the IV line',
        'Select a vein':                 'selecting a suitable vein for injection',
        'Remove needle':                 'safely removing the needle after the procedure',
        'Perform subcutaneous puncture': 'performing a subcutaneous puncture',
        'Measure blood glucose level':   'measuring the patient\'s blood glucose level',
        'Connect infusion device':       'connecting the infusion device to the IV line',
        'Prepare glucometer':            'preparing the glucometer for blood glucose testing',
        'Establish a sterile zone':      'establishing a sterile zone for the procedure',
        'Aspirate medication':           'aspirating medication into the syringe',
        'Adjust drip rate':              'adjusting the IV drip rate',
        'Put on isolation gown':         'putting on an isolation gown',
        'Assist patient taking medicine':'assisting the patient in taking medicine',
        'Cleanse lips':                  'cleansing the patient\'s lips',
        'Wear gloves':                   'putting on sterile gloves',
        'Select needle puncture site':   'selecting the needle puncture site',
    }
    action_phrase = action_to_phrase.get(action_name, f'performing {action_name.lower()}')
    return f"A nurse {action_phrase} as part of a {procedure_name.lower()} procedure in a hospital ward."

# 프레임으로 재구성
anno_lookup = {
    vid_id: item
    for vid_id, item in annotations.items()
    if item['operationID'] in PROCEDURE_MAP
}

dataset_items = []
for fp in sorted(FRAME_DIR.rglob('*.jpg')):
    parts    = fp.parts
    video_id = parts[-3]
    ann_dir  = parts[-2]

    if video_id not in anno_lookup:
        continue

    ann_idx = int(ann_dir.replace('ann', ''))
    item    = anno_lookup[video_id]
    op_id   = item['operationID']

    if ann_idx >= len(item['annotations']):
        continue

    ann         = item['annotations'][ann_idx]
    action_id   = ann['actionID']
    proc_name   = PROCEDURE_MAP[op_id]
    action_name = ACTION_MAP.get(action_id, f'action_{action_id}')

    dataset_items.append({
        'image_path': str(fp),
        'caption':    build_gt_caption(proc_name, action_name),
        'procedure':  proc_name,
        'action':     action_name,
        'video_id':   video_id,
        'action_id':  action_id,
        'segment':    ann['segment'],
    })

# actionID 21 필터링
dataset_items = [i for i in dataset_items if i['action_id'] != 21]
print(f'재구성: {len(dataset_items)}개')

# 저장
with open(DATA_DIR / 'dataset.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_items, f, ensure_ascii=False, indent=2)
print(f'저장 완료: {DATA_DIR / "dataset.json"}')

# split
random.seed(42)
random.shuffle(dataset_items)
n       = len(dataset_items)
n_train = int(n * 0.8)
n_val   = int(n * 0.1)

train_items = dataset_items[:n_train]
val_items   = dataset_items[n_train:n_train + n_val]
test_items  = dataset_items[n_train + n_val:]

print(f'Train: {len(train_items)}, Val: {len(val_items)}, Test: {len(test_items)}')

재구성: 1643개
저장 완료: /content/drive/MyDrive/Nurse/data/dataset.json
Train: 1314, Val: 164, Test: 165


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

collate_fn = partial(nursing_collate_fn, processor=ft_processor, device=DEVICE)

train_dataset = NursingCaptionDataset(train_items, ft_processor, NURSING_PROMPT)
val_dataset   = NursingCaptionDataset(val_items,   ft_processor, NURSING_PROMPT)

train_loader = DataLoader(train_dataset, batch_size=lora_cfg.batch_size,
                          shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=lora_cfg.batch_size,
                          shuffle=False, collate_fn=collate_fn, num_workers=0)

optimizer    = AdamW([p for p in ft_model.parameters() if p.requires_grad],
                     lr=lora_cfg.lr, weight_decay=lora_cfg.weight_decay)
total_steps  = len(train_loader) * lora_cfg.num_epochs // lora_cfg.grad_accum
scheduler    = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)

print(f'Train 배치 수: {len(train_loader)}, 총 스텝: {total_steps}')

Train 배치 수: 1314, 총 스텝: 246


In [ ]:
# ─── 학습 루프 ────────────────────────────────────────────────────────────
train_losses, val_losses = [], []
best_val_loss = float('inf')

for epoch in range(lora_cfg.num_epochs):

    # Train
    ft_model.train()
    epoch_loss = 0.0
    optimizer.zero_grad()
    pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                desc=f'Epoch {epoch+1}/{lora_cfg.num_epochs} [Train]')

    for step, batch in pbar:
        outputs = ft_model(**batch)
        loss    = outputs.loss / lora_cfg.grad_accum  # gradient accumulation
        loss.backward()
        epoch_loss += loss.item() * lora_cfg.grad_accum

        if (step + 1) % lora_cfg.grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(ft_model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        pbar.set_postfix({'loss': f'{epoch_loss/(step+1):.4f}'})

    avg_train = epoch_loss / len(train_loader)
    train_losses.append(avg_train)

    # Validation
    ft_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{lora_cfg.num_epochs} [Val]'):
            val_loss += ft_model(**batch).loss.item()

    avg_val = val_loss / len(val_loader)
    val_losses.append(avg_val)
    print(f'Epoch {epoch+1}: train_loss={avg_train:.4f}  val_loss={avg_val:.4f}')

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        ft_model.save_pretrained(lora_cfg.output_dir / 'best_lora')
        ft_processor.save_pretrained(lora_cfg.output_dir / 'best_lora')
        print(f'  → Best 모델 저장 (val_loss={best_val_loss:.4f})')

print('\n학습 완료!')

Epoch 1/3 [Train]:  24%|██▎       | 311/1314 [06:09<19:50,  1.19s/it, loss=0.6769]


KeyboardInterrupt: 

In [ ]:
# Loss 곡선 시각화
plt.figure(figsize=(8, 4))
plt.plot(range(1, lora_cfg.num_epochs+1), train_losses, 'b-o', label='Train Loss')
plt.plot(range(1, lora_cfg.num_epochs+1), val_losses,   'r-o', label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('LoRA Fine-tuning Loss Curve')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(lora_cfg.output_dir / 'loss_curve.png', dpi=150)
plt.show()

---
## 6. Fine-tuned 모델 평가

In [ ]:
from peft import PeftModel

# Best LoRA adapter 로드
base_for_eval, eval_processor = load_smolvlm(lora_cfg.model_id, DEVICE)
ft_model_eval = PeftModel.from_pretrained(base_for_eval, lora_cfg.output_dir / 'best_lora')
ft_model_eval.eval()
print('Fine-tuned 모델 로드 완료')

In [ ]:
# Fine-tuned Caption 생성 (동일한 test split)
ft_predictions = generate_captions_batch(ft_model_eval, eval_processor, test_items,
                                          prompt=NURSING_PROMPT, max_new_tokens=lora_cfg.max_new_tokens,
                                          device=DEVICE)
ft_metrics = compute_all_metrics(ft_predictions, zs_references)

print('\n=== Fine-tuned 평가 결과 ===')
for metric, score in ft_metrics.items():
    print(f'  {metric:10s}: {score:.4f}')

---
## 7. Before / After 비교 분석

In [ ]:
# ─── 정량 평가: 지표 비교 테이블 ─────────────────────────────────────────
print('=' * 55)
print(f'{"Metric":<12} {"Zero-shot":>12} {"LoRA FT":>12} {"향상":>12}')
print('=' * 55)
for metric in ['BLEU-1','BLEU-2','BLEU-3','BLEU-4','METEOR','CIDEr-D']:
    zs_v = zs_metrics.get(metric, 0)
    ft_v = ft_metrics.get(metric, 0)
    diff = ft_v - zs_v
    print(f'{metric:<12} {zs_v:>12.4f} {ft_v:>12.4f} {"+" if diff>=0 else ""}{diff:>11.4f}')
print('=' * 55)

In [ ]:
# 실패 분석 비교
ft_failure = analyze_failure(ft_predictions, zs_references)

print('\n=== 실패 분석 비교 ===')
print(f'{"항목":<30} {"Zero-shot":>10} {"LoRA FT":>10}')
print('-' * 55)
for key, label in [('medical_object_fail_pct','의료 Object 인식 실패(%)'),
                   ('medical_action_collapse_pct','의료 행동 Collapse(%)'),
                   ('hallucination_pct','Hallucination(%)')]:
    print(f'{label:<30} {zs_failure[key]:>9.1f}% {ft_failure[key]:>9.1f}%')

In [ ]:
# ─── 정성 평가: Before/After 이미지 시각화 ──────────────────────────────
N_COMPARE = min(4, len(test_items))
fig, axes = plt.subplots(N_COMPARE, 1, figsize=(14, 5*N_COMPARE))
if N_COMPARE == 1: axes = [axes]

for i, ax in enumerate(axes):
    ax.imshow(Image.open(test_items[i]['image_path']).convert('RGB'))
    ax.axis('off')
    ax.set_title(
        f"[{test_items[i]['procedure']}] {test_items[i]['action']}\n"
        f"GT:        {zs_references[i]}\n"
        f"Zero-shot: {zs_predictions[i]}\n"
        f"LoRA FT:   {ft_predictions[i]}",
        fontsize=8, loc='left'
    )

plt.suptitle('Before / After Caption 비교', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(lora_cfg.output_dir / 'before_after_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Metric Bar Chart ─────────────────────────────────────────────────────
metrics_to_plot = ['BLEU-1','BLEU-2','BLEU-4','METEOR','CIDEr-D']
zs_vals = [zs_metrics.get(m,0) for m in metrics_to_plot]
ft_vals = [ft_metrics.get(m,0) for m in metrics_to_plot]

x, width = np.arange(len(metrics_to_plot)), 0.35
fig, ax  = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x-width/2, zs_vals, width, label='Zero-shot', color='steelblue', alpha=0.8)
b2 = ax.bar(x+width/2, ft_vals, width, label='LoRA FT',   color='tomato',    alpha=0.8)

for bar in [*b1, *b2]:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x); ax.set_xticklabels(metrics_to_plot)
ax.set_ylabel('Score')
ax.set_title('SmolVLM Zero-shot vs LoRA Fine-tuned\nNursing Caption Metrics')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(lora_cfg.output_dir / 'metric_comparison.png', dpi=150)
plt.show()

In [ ]:
# ─── 최종 결과 JSON 저장 ──────────────────────────────────────────────────
final_results = {
    'dataset': {'total': len(dataset_items), 'train': len(train_items),
                'val': len(val_items), 'test': len(test_items),
                'procedures': list(PROCEDURE_MAP.values())},
    'model': lora_cfg.model_id,
    'lora_config': {'rank': lora_cfg.rank, 'alpha': lora_cfg.alpha,
                    'dropout': lora_cfg.dropout, 'target_modules': list(lora_cfg.target_modules),
                    'num_epochs': lora_cfg.num_epochs, 'lr': lora_cfg.lr},
    'zero_shot_metrics': zs_metrics,
    'lora_ft_metrics':   ft_metrics,
    'improvement':    {m: round(ft_metrics.get(m,0)-zs_metrics.get(m,0),4) for m in metrics_to_plot},
    'failure_analysis': {'zero_shot': zs_failure, 'lora_ft': ft_failure},
    'before_after_samples': [
        {'procedure': test_items[i]['procedure'], 'action': test_items[i]['action'],
         'gt': zs_references[i], 'zero_shot': zs_predictions[i], 'lora_ft': ft_predictions[i]}
        for i in range(min(10, len(test_items)))
    ],
    'train_losses': train_losses, 'val_losses': val_losses,
}

with open(lora_cfg.output_dir / 'final_results.json', 'w', encoding='utf-8') as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)

print('최종 결과 저장 완료')
print()
print('=== 최종 요약 ===')
for m in ['BLEU-4','METEOR','CIDEr-D']:
    zv, fv, dv = zs_metrics[m], ft_metrics[m], final_results['improvement'][m]
    print(f'{m:<10} Zero-shot: {zv:.4f}  →  LoRA FT: {fv:.4f}  (Δ{"+" if dv>=0 else ""}{dv:.4f})')